# WP4v5 — Notebook 1 : Génération

**Stratégie** : pour chaque image, générer 100 CLS MAE masqués (seeds 0..99),
puis créer K=50 représentations en moyennant 5 tirages aléatoires parmi les 100.

**Stockage optimisé** :
- `mae_avg` : (N×50, 1024) — représentations MAE moyennées, brutes
- `img_idx` : (N×50,) — index de l'image source (pour retrouver le CLS CLIP)
- `cls_clip` : (N, 1024) — **une seule fois par image** (pas 50 fois)
- `labels`   : (N,)

Total : ~27GB au lieu de ~53GB si on répétait cls_clip 50 fois.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import ViTImageProcessor, ViTMAEModel
from transformers import LlavaForConditionalGeneration, CLIPImageProcessor
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm
import numpy as np

DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
N_MASKS     = 50  # tirages masques par image
K_GROUPS    = 50   # groupes moyennes par image
GROUP_SIZE  = 5    # taille de chaque groupe

print(f'Device : {DEVICE}')
print(f'Paires generees par image : {K_GROUPS} (moyenne de {GROUP_SIZE} parmi {N_MASKS} masquages)')


/home/philippelin/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device : cuda
Paires generees par image : 50 (moyenne de 5 parmi 100 masquages)


In [ ]:
mae_processor = ViTImageProcessor(
    size={'height': 224, 'width': 224},
    image_mean=[0.485, 0.456, 0.406],
    image_std=[0.229, 0.224, 0.225],
)
mae_encoder = ViTMAEModel.from_pretrained('./vit-mae-large').to(DEVICE)
mae_encoder.eval()

llava = LlavaForConditionalGeneration.from_pretrained(
    './llava-1.5-7b-hf', torch_dtype=torch.float16
)
vision_tower = llava.vision_tower.to(DEVICE).eval()
del llava; torch.cuda.empty_cache()
clip_proc = CLIPImageProcessor.from_pretrained('./llava-1.5-7b-hf')
print(f'Modeles charges — VRAM : {torch.cuda.memory_allocated()/1e9:.1f} GB')


In [ ]:
ds_train = load_dataset('parquet', data_files={'train': './imagenet100/data/train-*.parquet'})
ds_val   = load_dataset('parquet', data_files={'validation': './imagenet100/data/validation-*.parquet'})

transform = transforms.Compose([
    transforms.Resize(384), transforms.CenterCrop(336), transforms.ToTensor(),
])

class HFImageDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.data = hf_dataset; self.transform = transform
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        img  = item['image'].convert('RGB')
        if self.transform: img = self.transform(img)
        return img, item['label']

dataset_train = HFImageDataset(ds_train['train'],    transform=transform)
dataset_val   = HFImageDataset(ds_val['validation'], transform=transform)
print(f'Train : {len(dataset_train)} images -> {len(dataset_train)*K_GROUPS} paires')
print(f'Val   : {len(dataset_val)} images -> {len(dataset_val)*K_GROUPS} paires')


In [ ]:
def generate_pairs(dataset, batch_size=32, desc=''):
    """
    Pour chaque image :
    1. Calculer CLS CLIP une fois
    2. Calculer N_MASKS=100 CLS MAE masques
    3. Creer K_GROUPS=50 representations en moyennant GROUP_SIZE=5 masquages aleatoires

    Retourne (BRUT, non normalise) :
    - mae_avg  : (N*K, 1024) — representations moyennees
    - cls_clip : (N, 1024)   — une par image, L2-normalise
    - img_idx  : (N*K,)      — index de l'image source
    - labels   : (N,)
    """
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=0, pin_memory=(DEVICE=='cuda'))

    all_mae_avg  = []
    all_cls_clip = []
    all_img_idx  = []
    all_labels   = []
    global_idx   = 0

    for images_336, labels in tqdm(loader, desc=desc):
        B = images_336.shape[0]
        images_224 = F.interpolate(images_336, size=(224,224),
                                   mode='bilinear', align_corners=False)

        # --- CLS CLIP (une seule fois par image) ---
        clip_in = clip_proc(images=list(images_336), return_tensors='pt', do_rescale=False)
        pix = clip_in['pixel_values'].to(DEVICE).half()
        with torch.no_grad():
            cls_clip_batch = vision_tower(pix).last_hidden_state[:, 0]  # (B, 1024)
        cls_clip_batch = F.normalize(cls_clip_batch.cpu().float(), dim=-1)

        # --- N_MASKS CLS MAE masques par image ---
        mae_in = mae_processor(images=list(images_224), return_tensors='pt', do_rescale=False)
        mae_in = {k: v.to(DEVICE) for k, v in mae_in.items()}

        cls_pool = []  # (N_MASKS, B, 1024)
        for seed in range(N_MASKS):
            gen   = torch.Generator().manual_seed(seed)
            noise = torch.rand(B, 196, generator=gen).to(DEVICE)
            with torch.no_grad():
                out = mae_encoder(**mae_in, noise=noise)
            cls_pool.append(out.last_hidden_state[:, 0].cpu().float())  # (B, 1024)

        cls_pool = torch.stack(cls_pool)  # (N_MASKS, B, 1024)

        # --- K_GROUPS moyennes aleatoires de GROUP_SIZE masquages ---
        mae_avg_batch = []  # (K_GROUPS, B, 1024)
        for _ in range(K_GROUPS):
            # Choisir GROUP_SIZE seeds aleatoirement parmi N_MASKS
            chosen = torch.randperm(N_MASKS)[:GROUP_SIZE]
            avg    = cls_pool[chosen].mean(dim=0)  # (B, 1024)
            mae_avg_batch.append(avg)

        # Restructurer : (K_GROUPS*B, 1024)
        mae_avg_batch = torch.cat(mae_avg_batch, dim=0)  # (K_GROUPS*B, 1024)

        # img_idx : pour chaque paire, quel est l'index de l'image source
        img_idx = torch.arange(global_idx, global_idx + B).repeat(K_GROUPS)  # (K_GROUPS*B,)

        all_mae_avg.append(mae_avg_batch)
        all_cls_clip.append(cls_clip_batch)   # (B, 1024) — une par image
        all_img_idx.append(img_idx)
        all_labels.append(labels)
        global_idx += B

    return {
        'mae_avg':  torch.cat(all_mae_avg),   # (N*K, 1024)
        'cls_clip': torch.cat(all_cls_clip),  # (N, 1024)
        'img_idx':  torch.cat(all_img_idx),   # (N*K,)
        'labels':   torch.cat(all_labels),    # (N,)
    }

data_train = generate_pairs(dataset_train, batch_size=32, desc='Train')
data_val   = generate_pairs(dataset_val,   batch_size=32, desc='Val')

torch.save(data_train, 'wp4v5_pairs_train.pt')
torch.save(data_val,   'wp4v5_pairs_val.pt')
print('Sauvegarde OK')
print(f'  mae_avg  : {data_train["mae_avg"].shape}')   # (N*K, 1024)
print(f'  cls_clip : {data_train["cls_clip"].shape}')  # (N, 1024)
print(f'  img_idx  : {data_train["img_idx"].shape}')   # (N*K,)


In [ ]:
# Stats de normalisation sur mae_avg du train
norm_mean = data_train['mae_avg'].mean(dim=0)
norm_std  = data_train['mae_avg'].std(dim=0).clamp(min=1e-6)
torch.save({'mean': norm_mean, 'std': norm_std}, 'wp4v5_norm_stats.pt')
print('Stats sauvegardees')

# Verification discriminabilite inter-images
idx5 = [i * K_GROUPS for i in range(5)]
vecs = F.normalize(data_val['mae_avg'][idx5], dim=-1)
sim  = (vecs @ vecs.T).numpy()
print('\nSimilarites cosinus mae_avg inter-images (groupe 0) :')
print(sim.round(4))

# Verification intra-image (5 groupes de la meme image)
vecs2 = F.normalize(data_val['mae_avg'][:5], dim=-1)  # 5 groupes image 0
sim2  = (vecs2 @ vecs2.T).numpy()
print('\nSimilarites cosinus mae_avg intra-image (5 groupes, image 0) :')
print(sim2.round(4))

del vision_tower, mae_encoder; torch.cuda.empty_cache()
